## What a model is useful for

- The same chain shape does very different jobs, changed only by the prompt
- Prompt, model, parser ; that pattern repeats for every task below

Six use cases in order : text generation, classification, document analysis,
translation, question answering and summarisation.

### Installing the libraries

In [ ]:
# (setup cell already installs what this notebook needs)

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

Text generation
1. Run the chain (`chain`) for 3 different queries and compare the length and structure of the responses.
2. Add a simple execution-time measurement (time.perf_counter) and print the results.
3. Implement a helper function `run_query(text: str) -> str` that calls the chain and returns just the response text.

In [ ]:
from langchain_openai import ChatOpenAI

def run_query(text: str) -> str:
    """Send a prompt to the LLM and return just the response text."""
    return llm.invoke(text).content

# create the client, pointed at local Ollama
llm = make_llm()

# simple prompt and request to the LLM API
response = llm.invoke("Write a two-line job advert for a warehouse planner.")
print("Model response:\n")
print(response.content)

### Exercise Text generation
Run the chain (`chain`) for 2 different queries and compare the length and structure of the responses.

In [ ]:
response = llm.invoke("Write a product description for a cordless barcode scanner.")

print("Model response:\n")
print(response.content)

### Solution 

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM on local Ollama; a little temperature since this is creative generation
llm = make_llm(temperature=0.7)

# Build the chain: prompt template -> model -> plain-text output
prompt = ChatPromptTemplate.from_template("{query}")
chain = prompt | llm | StrOutputParser()

# Two deliberately different queries (different structure: a recipe vs. a poem)
queries = [
    "Write a product description for a cordless barcode scanner.",
    "Write a short welcome note for someone starting on Monday.",
]

results = []
for i, q in enumerate(queries, start=1):
    text = chain.invoke({"query": q})
    results.append({
        "query": q,
        "characters": len(text),
        "words": len(text.split()),
        "lines": text.count("\n") + 1,
    })
    print(f"=== Query {i}: {q} ===")
    print(f"Characters: {len(text)} | Words: {len(text.split())} | Lines: {text.count(chr(10)) + 1}\n")
    print(text, "\n")

In [8]:
import pandas as pd

df = pd.DataFrame(results, index=["query_1", "query_2"])
df

,query,characters,words,lines
query_1,Generate a recipe for a sweet cheesecake with ...,120,20,1
query_2,Write a short poem about autumn in Amsterdam.,276,43,9


### Exercise Classification 
Modify the texts and the prompt in the code below so that it classifies the sentiment of the text (positive, negative, neutral). Include texts with different sentiments in the examples.
```
[
    "This product is amazing! I loved it.",
    "I am very disappointed. The product broke after one use.",
    "It's okay, does the job but nothing special."
]
```

In [ ]:
articles = [
    "The government announced new tax reforms today.",
    "The local team won the championship in a thrilling match.",
    "New advancements in AI are reshaping the tech industry.",
    "The art exhibit showcased contemporary works by emerging artists.",
    "New guidelines for a healthy diet were published by the health department."
]
response = llm.invoke('Classify the attached texts into the groups "Politics", "Sport", "Technology", "Culture", "Health".\n\n' + '\n'.join(articles))

print("Model response:\n")
print(response.content)

### Solution 
The task is to adapt the topic-classification code into a sentiment classifier: swap the texts for ones with positive/negative/neutral sentiment, and change the prompt to classify sentiment instead of topic. Here's the full solution on the Ollama bridge:

In [9]:
from langchain_openai import ChatOpenAI

llm = make_llm()

# Texts with clearly different sentiments (positive, negative, neutral)
texts = [
    "This product is amazing! I loved it.",
    "I am very disappointed. The product broke after one use.",
    "It's okay, does the job but nothing special.",
    "Best purchase I've made all year, highly recommend!",
    "The delivery was on time and the package was intact.",
]

prompt = (
    'Classify the sentiment of each text below as "positive", "negative", or '
    '"neutral". Return one line per text in the format: <number>. <sentiment>\n\n'
    + "\n".join(f"{i}. {t}" for i, t in enumerate(texts, start=1))
)

response = llm.invoke(prompt)
print("Model response:\n")
print(response.content)

Model response:

Here are the classifications:

1. Positive
2. Negative
3. Neutral
4. Positive
5. Neutral


### Exercise 4 Document analysis 
Run the chain for 3 different prompts:
1. For which years are the values presented in the report?
2. What is the value of "Netto-Cashflow" in 2019?
3. Find information about the company's annual revenue.

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
file = open(data("annual_report.html"))
document = file.read()
response = llm.invoke(f"Analyze the attached document ____ \n\n {document}")
print("Model response:\n")
print(response.content)
```

In [10]:
from langchain_openai import ChatOpenAI

llm = make_llm()

# Load the report once
with open(data("annual_report.html"), "r", encoding="utf-8") as file:
    document = file.read()

# The three questions from the exercise
questions = [
    "For which years are the values presented in the report?",
    'What is the value of "Netto-Cashflow" in 2019?',
    "Find information about the company's annual revenue.",
]

for i, question in enumerate(questions, start=1):
    response = llm.invoke(
        f"Analyze the attached document and answer this question: {question}\n\n{document}"
    )
    print(f"=== Question {i}: {question} ===")
    print(response.content, "\n")

=== Question 1: For which years are the values presented in the report? ===
The values presented in the report are for the years 2019 and 2018. The table shows a comparison of various financial metrics between these two years. 

=== Question 2: What is the value of "Netto-Cashflow" in 2019? ===
The value of "Netto-Cashflow" in 2019 is €3.160 million. This can be found in the last row of the table, where it says "Netto-Cashflow" and has a value of "3.160" in the column for 2019. 

=== Question 3: Find information about the company's annual revenue. ===
The company's annual revenue for 2019 is €55.680 million (as shown in the table under the row "Umsatzerlöse" and column "2019"). This is a decrease of -6.0% compared to the previous year, when the company's annual revenue was €59.248 million. 



### Exercise 5 Machine translation
Run the chain (`chain`) for 3 different queries, translating texts in different languages.
Does the model used know Swahili?

In [ ]:
response = llm.invoke("Translate the following text into Polish:\n I'm foreigner and I don't speak german fluently.")

print("Model response:\n")
print(response.content)

### Solution 

In [14]:
from langchain_openai import ChatOpenAI

llm = make_llm()

text = "I'm a foreigner and I don't speak German fluently."

# Three different target languages
languages = ["Polish", "French", "Swahili"]

for i, lang in enumerate(languages, start=1):
    response = llm.invoke(f"Translate the following text into {lang}:\n{text}")
    print(f"=== Query {i}: → {lang} ===")
    print(response.content, "\n")

=== Query 1: → Polish ===
Jestem obcokrajowcem i nie mówię po niemiecku płynnie. 

=== Query 2: → French ===
Here is the translation:

Je suis un étranger et je ne parle pas allemand avec aisance.

Note: "fluent" can be translated to French as "avec aisance", which means speaking with ease or facility. However, a more common way to express this idea in French would be "je ne parle pas très bien allemand". 

=== Query 3: → Swahili ===
Ninachukua nafasi ya mwanafunzi wa nje na siweze kusema Kijerumani kwa usahihi. 



### Exercise Question answering
Insert a question about the information in the provided prompt

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
response = llm.invoke("Answer the questions below based on the attached text:\n"
"LangChain is a framework for working with large language models.\n"
"Chains in LangChain are data flows between prompts, models and parsers.\n"
"A retriever lets you search for information in a vector store.\n"
"____") # <- Insert a question about the information in the provided prompt np. Czym jest LangChain?

print("Model response:\n")
print(response.content)
```

In [ ]:
from langchain_openai import ChatOpenAI

llm = make_llm()

context = (
    "LangChain is a framework for working with large language models.\n"
    "Chains in LangChain are data flows between prompts, models and parsers.\n"
    "A retriever lets you search for information in a vector store.\n"
)

question = "What is a service level agreement?"   # <- the inserted question

response = llm.invoke(
    f"Answer the question below based on the attached text:\n{context}\n{question}"
)
print("Model response:\n")
print(response.content)

### Explanation
Questions 1 and 2 are answered directly from the context. Question 3 is the teaching moment: the answer isn't in the three facts, so a well-behaved model should say "the text doesn't mention that". Without the ONLY ... if not in the text, say so instruction, the model will often answer anyway from its training data (LangChain was created by Harrison Chase in 2022), which looks helpful but is exactly the behaviour we don't want in a grounded question-answering system.


In [ ]:
questions = [
    "What is a service level agreement?",                        # answerable : fact is in the text
    "What does a retriever do?",                 # answerable : fact is in the text
    "Who created LangChain and in what year?",   # NOT in the text : watch what happens
]

for i, q in enumerate(questions, start=1):
    response = llm.invoke(
       f"Answer the question below based ONLY on the attached text. "
       f"If the answer is not in the text, say so.\n\n{context}\nQuestion: {q}"

    )
    print(f"=== Question {i}: {q} ===")
    print(response.content, "\n")

### Exercise Summarization

1. Upload a text file of our own : the folder icon in the left sidebar, then
   the upload button. Made up or public text, nothing from work
2. Put its name in the code below, in place of the one there
3. Add a word limit to the prompt and summarise the file

In [ ]:
file = open(data("nad_niemnem.txt"), "r", encoding="utf-8") # <- replace the file path here
document = file.read()
response = llm.invoke(f"Write a short summary (approx. 500 words) of the attached text.\n{document[:1800]}")

print("Model response:\n")
print(response.content)

### Solution 

In [22]:
from langchain_openai import ChatOpenAI

llm = make_llm()

# Step 2: replace with the path to your own document
with open(data("nad_niemnem.txt"), "r", encoding="utf-8") as file:
    document = file.read()

# Step 3: word limit is stated in the prompt
word_limit = 150

response = llm.invoke(
    f"Write a summary of the following text in no more than {word_limit} words.\n\n"
    f"{document[:4000]}"
)
print("Model response:\n")
print(response.content)

Model response:

Poniżej jest krótki podsumowanie tekstu.

Dzień był letni i świąteczny. Wszystko na świecie jaśniało, kwitło i pachniało. Równina była pełna życia: ptaki śpiewały, owady taneczkowały, a ludzie byli weseli. Wiejskie kobiety szły po drogach i miedzach, ubrane w czerwone i żółte chusty, tworząc korowody żywych piwonii i słoneczników. Ludzie rozmawiali, śmiali się, płakali i śpiewali pieśni. W tym ruchu ludzkim czuć było najpiękniejszy moment dla wiejskiej ludności: wesoły powrót z kościoła. Dwie kobiety ukazały się na równinie, szły one razem i były widoczne poza gromadami ludzi.


### Try our own text

- Swap in a document of our own for the summarisation task. Made up, not from work
- Classification is the one worth attention : it is the same call, and the
  prompt alone turns free text into a fixed set of labels